In [0]:
WITH job_usage AS (
      SELECT
          usage_metadata.job_id AS job_id,
          usage_metadata.cluster_id AS cluster_id,
          workspace_id,
          identity_metadata.run_as AS user,
          sku_name,
          usage_metadata.node_type AS node_type,
          count(DISTINCT usage_metadata.job_run_id) AS execution_count,
          sum(usage_quantity) AS total_usage
      FROM system.billing.usage
      WHERE usage_date >= current_date() - INTERVAL 7 DAYS
        AND usage_date <  current_date() - interval 1 day
        AND usage_metadata.job_id IS NOT NULL
        AND sku_name = 'PREMIUM_JOBS_COMPUTE'
      GROUP BY usage_metadata.job_id, usage_metadata.cluster_id, workspace_id,
              identity_metadata.run_as, sku_name, usage_metadata.node_type
  ),

  job_cost AS (
      SELECT
          n.job_id,
          n.cluster_id,
          n.workspace_id,
          n.user,
          n.execution_count,
          n.total_usage,
          m.Meter,
          ((n.total_usage / NULLIF(t.total_usage, 0)) * t.total_cost) AS total_cost
      FROM job_usage n
      JOIN value_stream_monitoring.meter_node_mapping m
          ON n.node_type = m.node_type
      JOIN prd.value_stream_monitoring.sku_usage_cost t
          ON m.Meter = t.sku
  ),

  per_cluster_util AS (
      SELECT
          nt.cluster_id,
          percentile_approx(nt.cpu_user_percent + nt.cpu_system_percent, 0.95) AS p95_max_cpu_pct,
          percentile_approx(nt.mem_used_percent, 0.95) AS p95_max_mem_used_pct,
          max_by(nt.node_type, nt.cpu_user_percent + nt.cpu_system_percent) AS node_type_at_max_cpu
      FROM system.compute.node_timeline nt
      WHERE nt.start_time >= current_date() - INTERVAL 30 DAYS
      GROUP BY nt.cluster_id
  )

  SELECT
      jc.job_id,
      jc.user,
      jc.workspace_id,
      sum(jc.total_cost) AS total_cost,
      sum(jc.execution_count) AS execution_count,
      MAX(pcu.p95_max_cpu_pct) AS worst_case_p95_cpu_pct,
      MAX(pcu.p95_max_mem_used_pct) AS worst_case_p95_mem_used_pct
  FROM job_cost jc
  JOIN per_cluster_util pcu
      ON jc.cluster_id = pcu.cluster_id
  GROUP BY jc.job_id, jc.user, jc.workspace_id, jc.Meter
  HAVING worst_case_p95_cpu_pct < 50 AND worst_case_p95_mem_used_pct < 50
  ORDER BY total_cost DESC